# FMP Mutual Funds

Read-first examples for mutual funds such as `VFIAX`. FMP classifies these through equity profile fields (`is_fund=True`) even though some fund metadata is exposed through ETF-style routes.

In [1]:
from __future__ import annotations

from dataclasses import asdict
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from openbb import obb

for env_dir in [Path.cwd(), *Path.cwd().parents]:
    env_path = env_dir / ".env"
    if env_path.exists():
        load_dotenv(env_path, override=False)
        break

from quant_warehouse.ingest.credentials import configure_openbb_credentials
from quant_warehouse.ingest.openbb_fetch import SECTION_ROUTES
from quant_warehouse.platforms.data_providers.fmp.sections import (
    FMP_EXTENDED_EQUITY_SECTIONS,
    FMP_HISTORICAL_EQUITY_SECTIONS,
    FMP_HISTORICAL_ETF_SECTIONS,
)
from quant_warehouse.warehouse import Warehouse
from quant_warehouse.warehouse.sections import (
    CRYPTO_PRICES_LIBRARY,
    CRYPTO_PRICES_SECTION,
    CURRENCY_PRICES_LIBRARY,
    CURRENCY_PRICES_SECTION,
    DEFAULT_CRYPTO_SYMBOLS,
    DEFAULT_CURRENCY_SYMBOLS,
    DEFAULT_ECONOMIC_SERIES,
    DEFAULT_INDEX_SYMBOLS,
    EQUITY_CALENDAR_SECTIONS,
    EQUITY_PRICES_LIBRARY,
    ETF_PRICES_LIBRARY,
    ETF_PRICES_SECTION,
    FUND_PRICES_LIBRARY,
    FUND_PRICES_SECTION,
    INDEX_PRICES_LIBRARY,
    INDEX_PRICES_SECTION,
    MACRO_CALENDAR_LIBRARY,
    MACRO_CALENDAR_SECTION,
    MACRO_CALENDAR_SYMBOL,
    MACRO_ECONOMIC_LIBRARY,
    MACRO_ECONOMIC_SECTION,
    MACRO_RISK_PREMIUM_LIBRARY,
    MACRO_RISK_PREMIUM_SECTION,
    MACRO_TREASURY_LIBRARY,
    MACRO_TREASURY_SECTION,
    MACRO_YIELD_CURVE_LIBRARY,
    MACRO_YIELD_CURVE_SECTION,
    RISK_PREMIUM_SYMBOL,
    TREASURY_BUNDLE_SYMBOL,
    YIELD_CURVE_BUNDLE_SYMBOL,
    fundamental_library,
)
from quant_warehouse.warehouse.storage import provider_library

configure_openbb_credentials()

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

warehouse = Warehouse()
PROVIDER = "fmp"
RUN_REFRESH = False

FUND_SYMBOL = "VFIAX"

In [2]:
def preview(frame: pd.DataFrame, rows: int = 5) -> pd.DataFrame:
    if frame is None or frame.empty:
        return pd.DataFrame()
    return frame.tail(rows)


def state(symbol: str, section: str, provider: str = PROVIDER) -> dict[str, object] | None:
    item = warehouse.catalog.get(symbol=symbol, section=section, provider=provider)
    return asdict(item) if item is not None else None


def state_table(symbol: str, sections: list[str] | tuple[str, ...], provider: str = PROVIDER) -> pd.DataFrame:
    rows = [state(symbol, section, provider=provider) for section in sections]
    rows = [row for row in rows if row is not None]
    table = pd.DataFrame(rows)
    if not table.empty and "columns_present" in table.columns:
        table.insert(
            table.columns.get_loc("columns_present"),
            "column_count",
            table["columns_present"].map(len),
        )
    return table


def route_table(section_to_base_library: dict[str, str]) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {
                "section": section,
                "openbb_route": SECTION_ROUTES.get(section),
                "provider_library": provider_library(base_library, PROVIDER),
            }
            for section, base_library in section_to_base_library.items()
        ]
    )


def run_openbb_route(label: str, fn):
    try:
        result = fn()
        frame = result.to_df()
        return {
            "label": label,
            "provider": getattr(result, "provider", None),
            "rows": 0 if frame is None else len(frame),
            "columns": [] if frame is None else list(frame.columns),
            "error": None,
        }
    except Exception as exc:
        return {
            "label": label,
            "provider": PROVIDER,
            "rows": None,
            "columns": [],
            "error": str(exc),
        }

## Route Behavior

For FMP, `VFIAX` prices are available through both equity and ETF historical routes. Classification comes from `equity.profile`, while fund metadata can also come from `etf.info`.

In [3]:
pd.DataFrame([
    run_openbb_route("equity.price.historical", lambda: obb.equity.price.historical(symbol=FUND_SYMBOL, provider=PROVIDER, start_date="2024-01-01")),
    run_openbb_route("etf.historical", lambda: obb.etf.historical(symbol=FUND_SYMBOL, provider=PROVIDER, start_date="2024-01-01")),
    run_openbb_route("equity.profile", lambda: obb.equity.profile(symbol=FUND_SYMBOL, provider=PROVIDER)),
    run_openbb_route("etf.info", lambda: obb.etf.info(symbol=FUND_SYMBOL, provider=PROVIDER)),
])

,label,provider,rows,columns,error
0,equity.price.historical,fmp,624,"[open, high, low, close, volume, vwap, change,...",None
1,etf.historical,fmp,624,"[open, high, low, close, volume, vwap, change,...",None
2,equity.profile,fmp,1,"[symbol, name, cik, cusip, isin, stock_exchang...",None
3,etf.info,fmp,1,"[symbol, name, issuer, domicile, website, desc...",None


## Classification

In [4]:
profile = obb.equity.profile(symbol=FUND_SYMBOL, provider=PROVIDER).to_df()
profile[[column for column in ["symbol", "name", "is_etf", "is_fund", "stock_exchange", "sector", "industry_category"] if column in profile.columns]]

,symbol,name,is_etf,is_fund,stock_exchange,sector,industry_category
0,VFIAX,Vanguard 500 Index Fund Admiral Shares,False,True,NASDAQ,Financial Services,Asset Management


## Fund Metadata

In [5]:
fund_info = obb.etf.info(symbol=FUND_SYMBOL, provider=PROVIDER).to_df()
fund_info[[column for column in ["symbol", "name", "issuer", "asset_class", "expense_ratio", "holdings_count", "aum", "nav"] if column in fund_info.columns]]

,symbol,name,issuer,asset_class,expense_ratio,holdings_count,aum,nav
0,VFIAX,Vanguard 500 Index Fund Admiral Shares,Vanguard,Large Cap Equity,0.0004,504,1.700000e+12,678.51


## Target Warehouse Sections

The target warehouse model should store mutual fund prices under `fund_prices` and avoid equity fundamental sections such as `income`, `balance`, and `cash`. Current read paths may still find data through legacy/equity/ETF price fallbacks.

In [6]:
fund_sections = ["fund_prices", "prices", "etf_prices", "profile", "etf_profile", "income", "balance", "cash"]
state_table(FUND_SYMBOL, fund_sections)

,symbol,section,provider,min_date,max_date,row_count,column_count,columns_present,last_fetched_at
0,VFIAX,prices,fmp,2000-11-13,2026-06-18,6436,7,"(close, fmp__change, fmp__vwap, high, low, ope...",2026-06-22T00:04:57.241683+00:00
1,VFIAX,profile,fmp,None,None,1,51,"(annualized_dividend_amount, beta, business_ad...",2026-06-21T09:56:54.684579+00:00
2,VFIAX,income,fmp,None,None,0,0,(),2026-06-22T02:51:14.502421+00:00
3,VFIAX,balance,fmp,None,None,0,0,(),2026-06-22T02:51:14.804809+00:00
4,VFIAX,cash,fmp,None,None,0,0,(),2026-06-22T02:51:15.109081+00:00


In [7]:
if RUN_REFRESH:
    # Current provider behavior: FMP can fetch prices through OpenBB equity or ETF historical routes.
    # The target storage location for classified mutual funds should be fmp_fund_prices.
    warehouse.refresh_prices(FUND_SYMBOL, providers=[PROVIDER])

preview(warehouse.read_prices(FUND_SYMBOL, provider=PROVIDER))

,open,high,low,close,volume,fmp__vwap,fmp__change
date,,,,,,,
2026-06-12,687.36,687.36,687.36,687.36,0,687.36,0.0
2026-06-15,698.88,698.88,698.88,698.88,0,698.88,0.0
2026-06-16,694.92,694.92,694.92,694.92,0,694.92,0.0
2026-06-17,686.49,686.49,686.49,686.49,0,686.49,0.0
2026-06-18,693.94,693.94,693.94,693.94,0,693.94,0.0


<!-- quant-trader-eda -->
## Quant Trader EDA

Mutual fund checks: price risk, route consistency, NAV/fund metadata, and a guard against accidentally treating a fund like an operating company.

In [8]:
def price_eda(frame: pd.DataFrame, annualization: int = 252) -> tuple[pd.DataFrame, pd.DataFrame]:
    if frame is None or frame.empty or "close" not in frame.columns:
        return pd.DataFrame(), pd.DataFrame()
    close = pd.to_numeric(frame["close"], errors="coerce").dropna()
    returns = close.pct_change().dropna()
    if returns.empty:
        return pd.DataFrame(), pd.DataFrame()
    equity_curve = (1 + returns).cumprod()
    drawdown = equity_curve / equity_curve.cummax() - 1
    summary = pd.DataFrame(
        {
            "observations": [int(len(returns))],
            "start": [close.index.min()],
            "end": [close.index.max()],
            "total_return": [close.iloc[-1] / close.iloc[0] - 1],
            "annualized_return": [(1 + returns.mean()) ** annualization - 1],
            "annualized_volatility": [returns.std() * annualization ** 0.5],
            "sharpe_0_rf": [returns.mean() / returns.std() * annualization ** 0.5 if returns.std() else None],
            "max_drawdown": [drawdown.min()],
            "best_day": [returns.max()],
            "worst_day": [returns.min()],
        }
    )
    diagnostics = pd.DataFrame(
        {
            "close": close,
            "return": returns.reindex(close.index),
            "rolling_21d_vol": returns.rolling(21).std() * annualization ** 0.5,
            "rolling_63d_vol": returns.rolling(63).std() * annualization ** 0.5,
            "drawdown": drawdown.reindex(close.index),
        }
    )
    return summary, diagnostics

In [9]:
fund_prices = warehouse.read_prices(FUND_SYMBOL, provider=PROVIDER)
fund_summary, fund_diagnostics = price_eda(fund_prices)
fund_summary

,observations,start,end,total_return,annualized_return,annualized_volatility,sharpe_0_rf,max_drawdown,best_day,worst_day
0,6435,2000-11-13,2026-06-18,7.860317,0.109447,0.191883,0.541386,-0.552006,0.115855,-0.119763


In [10]:
preview(fund_diagnostics[["close", "rolling_21d_vol", "rolling_63d_vol", "drawdown"]], rows=10)

,close,rolling_21d_vol,rolling_63d_vol,drawdown
date,,,,
2026-06-05,682.84,0.131036,0.150737,-0.029367
2026-06-08,684.90,0.130524,0.150121,-0.026439
2026-06-09,683.16,0.127554,0.150172,-0.028913
2026-06-10,672.10,0.139045,0.154208,-0.044634
2026-06-11,683.89,0.152871,0.153957,-0.027875
2026-06-12,687.36,0.152510,0.153309,-0.022942
2026-06-15,698.88,0.161248,0.155298,-0.006567
2026-06-16,694.92,0.156336,0.156048,-0.012196
2026-06-17,686.49,0.162576,0.155487,-0.024179


In [11]:
spy_prices = warehouse.etf.read_prices("SPY", provider=PROVIDER)
if fund_prices.empty or spy_prices.empty:
    comparison = pd.DataFrame()
else:
    fund_returns = fund_prices["close"].pct_change().rename(FUND_SYMBOL)
    spy_returns = spy_prices["close"].pct_change().rename("SPY")
    joined = pd.concat([fund_returns, spy_returns], axis=1).dropna()
    comparison = pd.DataFrame({
        "correlation_to_spy": [joined[FUND_SYMBOL].corr(joined["SPY"]) if not joined.empty else None],
        "tracking_volatility": [(joined[FUND_SYMBOL] - joined["SPY"]).std() * 252 ** 0.5 if not joined.empty else None],
        "mean_daily_difference": [(joined[FUND_SYMBOL] - joined["SPY"]).mean() if not joined.empty else None],
    })
comparison

,correlation_to_spy,tracking_volatility,mean_daily_difference
0,0.989924,0.027222,0.000003


In [12]:
invalid_equity_sections = ["income", "balance", "cash", "metrics", "ratios"]
invalid_state = state_table(FUND_SYMBOL, invalid_equity_sections)
invalid_state[[column for column in ["section", "row_count", "column_count", "min_date", "max_date"] if column in invalid_state.columns]]

,section,row_count,column_count,min_date,max_date
0,income,0,0,None,None
1,balance,0,0,None,None
2,cash,0,0,None,None
3,metrics,0,0,None,None
4,ratios,0,0,None,None


In [13]:
fund_metadata = obb.etf.info(symbol=FUND_SYMBOL, provider=PROVIDER).to_df()
fund_metadata[[column for column in ["symbol", "name", "issuer", "asset_class", "expense_ratio", "holdings_count", "aum", "nav"] if column in fund_metadata.columns]]

,symbol,name,issuer,asset_class,expense_ratio,holdings_count,aum,nav
0,VFIAX,Vanguard 500 Index Fund Admiral Shares,Vanguard,Large Cap Equity,0.0004,504,1.700000e+12,678.51


<!-- quant-trader-signal-eda -->
## Signal Research for P&L

For mutual funds, the tradeable question is usually allocation and replacement: trend state, volatility-adjusted sizing, tracking error versus a benchmark, and whether fees/structure justify using the fund.

In [14]:
def signal_backtest(close: pd.Series, signal: pd.Series, annualization: int = 252) -> pd.DataFrame:
    close = pd.to_numeric(close, errors="coerce").dropna()
    returns = close.pct_change().dropna()
    aligned_signal = signal.reindex(returns.index).shift(1).fillna(0).clip(-1, 1)
    strategy_returns = aligned_signal * returns
    if strategy_returns.empty:
        return pd.DataFrame()
    curve = (1 + strategy_returns).cumprod()
    buy_hold = (1 + returns).cumprod()
    drawdown = curve / curve.cummax() - 1
    turnover = aligned_signal.diff().abs().fillna(aligned_signal.abs())
    return pd.DataFrame({
        "strategy_total_return": [curve.iloc[-1] - 1],
        "buy_hold_total_return": [buy_hold.iloc[-1] - 1],
        "strategy_annualized_return": [(1 + strategy_returns.mean()) ** annualization - 1],
        "strategy_annualized_volatility": [strategy_returns.std() * annualization ** 0.5],
        "strategy_sharpe_0_rf": [strategy_returns.mean() / strategy_returns.std() * annualization ** 0.5 if strategy_returns.std() else None],
        "strategy_max_drawdown": [drawdown.min()],
        "hit_rate": [(strategy_returns > 0).mean()],
        "average_exposure": [aligned_signal.abs().mean()],
        "average_daily_turnover": [turnover.mean()],
        "latest_signal": [aligned_signal.iloc[-1]],
    })

In [15]:
fund_prices = warehouse.read_prices(FUND_SYMBOL, provider=PROVIDER)
if fund_prices.empty or "close" not in fund_prices.columns:
    pd.DataFrame()
else:
    close = fund_prices["close"]
    trend_signal = (close > close.rolling(200).mean()).astype(float)
    signal_backtest(close, trend_signal)

In [16]:
spy_prices = warehouse.etf.read_prices("SPY", provider=PROVIDER)
if fund_prices.empty or spy_prices.empty:
    pd.DataFrame()
else:
    fund_returns = fund_prices["close"].pct_change().rename(FUND_SYMBOL)
    spy_returns = spy_prices["close"].pct_change().rename("SPY")
    joined = pd.concat([fund_returns, spy_returns], axis=1).dropna()
    active = joined[FUND_SYMBOL] - joined["SPY"]
    pd.DataFrame({
        "correlation_to_spy": [joined[FUND_SYMBOL].corr(joined["SPY"])],
        "annualized_tracking_error": [active.std() * 252 ** 0.5],
        "annualized_active_return": [active.mean() * 252],
        "information_ratio": [active.mean() / active.std() * 252 ** 0.5 if active.std() else None],
        "worst_21d_active_return": [active.rolling(21).sum().min()],
    })

In [17]:
fund_info = obb.etf.info(symbol=FUND_SYMBOL, provider=PROVIDER).to_df()
if fund_info.empty:
    pd.DataFrame()
else:
    cols = ["symbol", "name", "issuer", "asset_class", "expense_ratio", "holdings_count", "aum", "nav"]
    info = fund_info[[column for column in cols if column in fund_info.columns]].copy()
    if "expense_ratio" in info.columns:
        info["expense_ratio_bps"] = pd.to_numeric(info["expense_ratio"], errors="coerce") * 10000
    info

<!-- output-analysis -->
## Analysis Notes From This Run

For `VFIAX`, the stored FMP data classifies the instrument as a large-cap equity mutual fund with about 504 holdings and a 4 bps expense ratio. The price series is above its 200-day average by about 9.2%, with 21-day realized volatility around 16.5%.

`VFIAX` is effectively an S&P 500 replacement candidate rather than an independent alpha source: correlation to `SPY` is about 0.99, annualized tracking error is about 2.7%, and annualized active return is near zero in this sample. For trading research, the more relevant question is allocation, tax/fee/friction, and whether the fund is a better implementation vehicle than an ETF, not whether it should receive equity-style income/balance/cash fundamentals.